# Wine Quality (Red): Revised Augmentation Protocol
Run with **Runtime → Run all** in Google Colab. The first run uses `N_RUNS = 1`; after checking output CSVs, change it to 10.
Outputs are saved in the same Google Drive folder as Credit Card and Forest Cover.


In [1]:
%pip -q install imbalanced-learn


In [2]:
"""Wine Quality (Red) — revised augmentation protocol.
Run in Google Colab (GPU recommended). Set N_RUNS=1 for the pilot, then 10.
Dependencies: numpy, pandas, scipy, scikit-learn, imbalanced-learn, torch.
"""
import os, gc, random, json, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE
import sklearn, imblearn
import torch
import torch.nn as nn
import torch.optim as optim

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
OUT_DIR = "/content/drive/MyDrive/CGA Ex/revision_protocol"
BASE_SEED = 44
N_RUNS = 10 # Pilot: change to 10 only after verifying outputs.
LAMBDA_GRID = [0.25, 0.50, 0.75, 1.00]
TEST_SIZE, VAL_SIZE = 0.20, 0.20
LATENT_DIM, BATCH_SIZE = 64, 256
EPOCHS_CGAN, EPOCHS_WGAN = 300, 300
N_CRITIC, GP_WEIGHT = 5, 10.0
FIDELITY_N_PER_CLASS = 1000
class_names = ["Low (3-4)", "Medium (5-6)", "High (7-8)"]
n_classes = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def setup_output():
    if os.path.isdir("/content"):
        from google.colab import drive
        drive.mount("/content/drive")
    os.makedirs(OUT_DIR, exist_ok=True)
    return Path(OUT_DIR)

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def load_data():
    df = pd.read_csv(DATA_URL, sep=";")
    assert df.shape[1] == 12 and df["quality"].notna().all()
    assert set(df["quality"].unique()).issubset({3, 4, 5, 6, 7, 8})
    y = np.select([df.quality <= 4, df.quality <= 6], [0, 1], default=2).astype(int)
    X = df.drop(columns=["quality"]).to_numpy(dtype=np.float32)
    assert np.isfinite(X).all()
    return X, y, df

def split_scale(X, y, seed):
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=VAL_SIZE/(1-TEST_SIZE),
        stratify=y_tv, random_state=seed+10000)
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)
    return X_train, X_val, X_test, y_train, y_val, y_test

def target_counts(y, strength):
    counts = pd.Series(y).value_counts().sort_index()
    nmax = int(counts.max())
    return {int(c): int(round(n + strength*(nmax-n))) for c,n in counts.items()}

def get_classifiers(seed):
    return {
        "RF": RandomForestClassifier(n_estimators=150, random_state=seed, n_jobs=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(64,64), max_iter=300, random_state=seed)
    }

def evaluate(Xtr, ytr, Xval, yval, Xtest, ytest, seed):
    metrics, recalls, validation = [], [], []
    for name, clf in get_classifiers(seed).items():
        clf.fit(Xtr, ytr)
        for Xev, yev, bucket in [(Xtest, ytest, metrics), (Xval, yval, validation)]:
            pred = clf.predict(Xev)
            bucket.append({
                "Classifier": name,
                "Macro_F1": f1_score(yev, pred, labels=range(n_classes), average="macro", zero_division=0),
                "Macro_Recall": recall_score(yev, pred, labels=range(n_classes), average="macro", zero_division=0),
                "Balanced_Accuracy": balanced_accuracy_score(yev, pred),
                "Accuracy": accuracy_score(yev, pred)
            })
            if bucket is metrics:
                for cls, val in enumerate(recall_score(yev, pred, labels=range(n_classes), average=None, zero_division=0)):
                    recalls.append({"Classifier": name, "Class_ID": cls,
                                    "Class": class_names[cls], "Recall": float(val)})
    return pd.DataFrame(metrics), pd.DataFrame(recalls), pd.DataFrame(validation)

def random_oversample_to_targets(X, y, targets, seed):
    rng = np.random.default_rng(seed)
    xs, ys = [X], [y]
    for cls, target in targets.items():
        idx = np.flatnonzero(y == cls)
        nadd = max(0, target-len(idx))
        if nadd:
            xs.append(X[rng.choice(idx, nadd, replace=True)])
            ys.append(np.full(nadd, cls, dtype=int))
    return np.vstack(xs), np.concatenate(ys)

def smote_to_targets(X, y, targets, seed):
    strategy = {c: n for c,n in targets.items() if n > np.sum(y==c)}
    if not strategy:
        return X.copy(), y.copy()
    return SMOTE(sampling_strategy=strategy, random_state=seed, k_neighbors=5).fit_resample(X,y)

def mean_marginal_w1(real, synth):
    if len(real)==0 or len(synth)==0:
        return float("nan")
    return float(np.mean([wasserstein_distance(real[:,j], synth[:,j]) for j in range(real.shape[1])]))

class Generator(nn.Module):
    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(n_classes, n_classes)
        self.net = nn.Sequential(
            nn.Linear(LATENT_DIM + n_classes, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, feature_dim), nn.Sigmoid()
        )
    def forward(self, z, y):
        return self.net(torch.cat([z, self.emb(y)], dim=1))

class Discriminator(nn.Module):
    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(n_classes, n_classes)
        self.net = nn.Sequential(
            nn.Linear(feature_dim + n_classes, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
        return self.net(torch.cat([x, self.emb(y)], dim=1))

class Critic(nn.Module):
    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.emb = nn.Embedding(n_classes, n_classes)
        self.net = nn.Sequential(
            nn.Linear(feature_dim + n_classes, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x, y):
        return self.net(torch.cat([x, self.emb(y)], dim=1))

def balanced_batch(Xt, yt, batch_size):
    per = max(1, batch_size // n_classes)
    xs, ys = [], []
    for cls in range(n_classes):
        idx = torch.where(yt == cls)[0]
        choose = idx[torch.randint(0, len(idx), (per,), device=DEVICE)]
        xs.append(Xt[choose]); ys.append(yt[choose])
    x = torch.cat(xs); lab = torch.cat(ys)
    if len(lab) < batch_size:
        extra = torch.randint(0, len(yt), (batch_size-len(lab),), device=DEVICE)
        x = torch.cat([x, Xt[extra]]); lab = torch.cat([lab, yt[extra]])
    perm = torch.randperm(len(lab), device=DEVICE)
    return x[perm], lab[perm]

def train_cgan(X_train, y_train, seed):
    set_seed(seed)
    Xt = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    G = Generator(X_train.shape[1], n_classes).to(DEVICE)
    D = Discriminator(X_train.shape[1], n_classes).to(DEVICE)
    optG = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
    optD = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
    bce = nn.BCELoss()

    for _ in range(EPOCHS_CGAN):
        real, labels = balanced_batch(Xt, yt, BATCH_SIZE)
        z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
        fake = G(z, labels)

        optD.zero_grad()
        dloss = bce(D(real, labels), torch.ones((len(labels),1), device=DEVICE)) + \
                bce(D(fake.detach(), labels), torch.zeros((len(labels),1), device=DEVICE))
        dloss.backward(); optD.step()

        z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
        optG.zero_grad()
        fake = G(z, labels)
        gloss = bce(D(fake, labels), torch.ones((len(labels),1), device=DEVICE))
        gloss.backward(); optG.step()
    return G

def gradient_penalty(C, real, fake, labels):
    eps = torch.rand(len(real), 1, device=DEVICE).expand_as(real)
    interp = eps*real + (1-eps)*fake
    interp.requires_grad_(True)
    score = C(interp, labels)
    grad = torch.autograd.grad(
        outputs=score, inputs=interp, grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    return ((grad.view(len(real), -1).norm(2, dim=1)-1)**2).mean()

def train_cwgan_gp(X_train, y_train, seed):
    set_seed(seed)
    Xt = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
    G = Generator(X_train.shape[1], n_classes).to(DEVICE)
    C = Critic(X_train.shape[1], n_classes).to(DEVICE)
    optG = optim.Adam(G.parameters(), lr=1e-4, betas=(0.5,0.9))
    optC = optim.Adam(C.parameters(), lr=1e-4, betas=(0.5,0.9))

    for _ in range(EPOCHS_WGAN):
        for _ in range(N_CRITIC):
            real, labels = balanced_batch(Xt, yt, BATCH_SIZE)
            z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
            fake = G(z, labels)
            optC.zero_grad()
            lossC = C(fake.detach(), labels).mean() - C(real, labels).mean()
            lossC = lossC + GP_WEIGHT*gradient_penalty(C, real, fake.detach(), labels)
            lossC.backward(); optC.step()

        _, labels = balanced_batch(Xt, yt, BATCH_SIZE)
        z = torch.randn(len(labels), LATENT_DIM, device=DEVICE)
        optG.zero_grad()
        fake = G(z, labels)
        lossG = -C(fake, labels).mean()
        lossG.backward(); optG.step()
    return G

def generate_by_counts(G, counts_to_generate):
    G.eval()
    xs, ys = [], []
    with torch.no_grad():
        for cls, n in counts_to_generate.items():
            if n <= 0:
                continue
            labels = torch.full((n,), int(cls), dtype=torch.long, device=DEVICE)
            z = torch.randn(n, LATENT_DIM, device=DEVICE)
            x = G(z, labels).cpu().numpy()
            xs.append(np.clip(x, 0.0, 1.0))
            ys.append(np.full(n, cls))
    if not xs:
        return np.empty((0, 11), np.float32), np.empty((0,), int)
    return np.vstack(xs), np.concatenate(ys)


def main():
    out = setup_output()
    X_raw, y, df = load_data()
    print("Wine Quality (Red)", X_raw.shape, "class counts", dict(zip(*np.unique(y, return_counts=True))))
    print("Device:", DEVICE, "Runs:", N_RUNS)
    assert set(np.unique(y)) == {0,1,2}
    manifest = {
        "dataset": "UCI Wine Quality (Red)", "source_url": DATA_URL,
        "label_definition": {"0": "quality 3-4", "1": "quality 5-6", "2": "quality 7-8"},
        "features": list(df.drop(columns=["quality"]).columns),
        "n_rows": len(df), "class_counts": pd.Series(y).value_counts().sort_index().to_dict(),
        "base_seed": BASE_SEED, "n_runs": N_RUNS, "lambda_grid": LAMBDA_GRID,
        "split": {"train": .6, "validation": .2, "test": .2},
        "selection": "Within each seed/method/classifier maximize validation Macro_F1; ties choose smaller lambda.",
        "fidelity": "Mean normalized univariate marginal W1; diagnostic only, not joint W1 upper bound.",
        "models": {"RF_trees": 150, "MLP_hidden": [64,64], "cgan_epochs": EPOCHS_CGAN,
                   "cwgan_epochs": EPOCHS_WGAN, "latent_dim": LATENT_DIM,
                   "batch_size": BATCH_SIZE, "critic_steps": N_CRITIC},
        "versions": {"python": sys.version, "numpy": np.__version__, "pandas": pd.__version__,
                     "sklearn": sklearn.__version__, "imblearn": imblearn.__version__,
                     "torch": torch.__version__}
    }
    (out/"wine_protocol_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
    metric_rows, class_rows, fidelity_rows, validation_rows = [], [], [], []
    def add_evaluation(Xaug, yaug, method, lam, run, seed, Xval, yval, Xtest, ytest):
        mt, cr, vl = evaluate(Xaug, yaug, Xval, yval, Xtest, ytest, seed)
        for tab in (mt, cr, vl):
            tab["RunID"], tab["Seed"], tab["Method"], tab["Balancing_Strength"] = run, seed, method, lam
        metric_rows.append(mt); class_rows.append(cr); validation_rows.append(vl)

    for run in range(N_RUNS):
        seed = BASE_SEED + run
        set_seed(seed)
        Xtrain, Xval, Xtest, ytrain, yval, ytest = split_scale(X_raw, y, seed)
        train_counts = pd.Series(ytrain).value_counts().sort_index().to_dict()
        assert min(train_counts.values()) >= 6
        add_evaluation(Xtrain, ytrain, "Real", 0.0, run, seed, Xval, yval, Xtest, ytest)
        print(f"Run {run+1}/{N_RUNS}: CGAN", flush=True)
        cgan = train_cgan(Xtrain, ytrain, seed)
        print(f"Run {run+1}/{N_RUNS}: CWGAN-GP", flush=True)
        cwgan = train_cwgan_gp(Xtrain, ytrain, seed)
        fixed_d = {}
        for method, G in [("CGAN",cgan), ("CWGAN-GP",cwgan)]:
            for cls in range(n_classes):
                nfid = max(FIDELITY_N_PER_CLASS, int(np.sum(yval==cls)))
                Xsyn, ysyn = generate_by_counts(G, {cls:nfid})
                fixed_d[(method,cls)] = mean_marginal_w1(Xval[yval==cls], Xsyn[ysyn==cls])
        for lam in LAMBDA_GRID:
            targets = target_counts(ytrain, lam)
            Xros, yros = random_oversample_to_targets(Xtrain,ytrain,targets,seed)
            add_evaluation(Xros,yros,"ROS",lam,run,seed,Xval,yval,Xtest,ytest)
            Xsm, ysm = smote_to_targets(Xtrain,ytrain,targets,seed)
            add_evaluation(Xsm,ysm,"SMOTE",lam,run,seed,Xval,yval,Xtest,ytest)
            gen_counts = {cls:max(0,targets[cls]-train_counts[cls]) for cls in targets}
            for method,G in [("CGAN",cgan), ("CWGAN-GP",cwgan)]:
                Xsyn, ysyn = generate_by_counts(G,gen_counts)
                Xaug = np.vstack([Xtrain,Xsyn])
                yaug = np.concatenate([ytrain,ysyn])
                add_evaluation(Xaug,yaug,method,lam,run,seed,Xval,yval,Xtest,ytest)
                n_aug = len(yaug)
                for cls in range(n_classes):
                    nr, ng = int(train_counts[cls]), int(gen_counts[cls])
                    alpha = ng/(nr+ng)
                    pi = (nr+ng)/n_aug
                    d = fixed_d[(method,cls)]
                    fidelity_rows.append({
                        "RunID":run,"Seed":seed,"Method":method,
                        "Balancing_Strength":lam,"Class_ID":cls,"Class":class_names[cls],
                        "n_real_train":nr,"n_generated":ng,"alpha_y":alpha,
                        "pi_tilde_y":pi,"Mean_Marginal_W1":d,
                        "Weighted_W1_Component":pi*alpha*d
                    })
        pd.concat(metric_rows,ignore_index=True).to_csv(out/"wine_metrics_checkpoint.csv",index=False)
        pd.concat(class_rows,ignore_index=True).to_csv(out/"wine_class_recall_checkpoint.csv",index=False)
        pd.DataFrame(fidelity_rows).to_csv(out/"wine_fidelity_checkpoint.csv",index=False)
        pd.concat(validation_rows,ignore_index=True).to_csv(out/"wine_validation_checkpoint.csv",index=False)
        print(f"Completed seed {seed}", flush=True)
        del cgan,cwgan
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    metrics = pd.concat(metric_rows,ignore_index=True)
    class_recall = pd.concat(class_rows,ignore_index=True)
    fidelity = pd.DataFrame(fidelity_rows)
    validation = pd.concat(validation_rows,ignore_index=True)
    delta = (fidelity.groupby(["RunID","Seed","Method","Balancing_Strength"],as_index=False)
             ["Weighted_W1_Component"].sum().rename(
             columns={"Weighted_W1_Component":"Delta_Component_over_L"}))
    candidates = validation[validation.Method != "Real"].copy()
    selected = (candidates.sort_values(
        ["RunID","Method","Classifier","Macro_F1","Balancing_Strength"],
        ascending=[True,True,True,False,True],kind="stable")
        .drop_duplicates(["RunID","Method","Classifier"])
        .rename(columns={"Balancing_Strength":"Selected_Lambda","Macro_F1":"Val_Macro_F1",
                         "Macro_Recall":"Val_Macro_Recall","Accuracy":"Val_Accuracy",
                         "Balanced_Accuracy":"Val_Balanced_Accuracy"}))
    selected = selected[["RunID","Seed","Method","Classifier","Selected_Lambda",
                         "Val_Macro_F1","Val_Macro_Recall","Val_Balanced_Accuracy","Val_Accuracy"]]
    selected_test = selected.merge(metrics,
        left_on=["RunID","Seed","Method","Classifier","Selected_Lambda"],
        right_on=["RunID","Seed","Method","Classifier","Balancing_Strength"],
        how="left",validate="one_to_one")
    assert len(metrics)==N_RUNS*34
    assert len(fidelity)==N_RUNS*2*4*n_classes
    assert len(validation)==N_RUNS*34
    assert len(selected)==N_RUNS*8 and selected_test.Macro_F1.notna().all()
    selected_summary = selected_test.groupby(["Method","Classifier"],as_index=False).agg(
        Macro_F1_mean=("Macro_F1","mean"),Macro_F1_sd=("Macro_F1","std"),
        Macro_Recall_mean=("Macro_Recall","mean"),Macro_Recall_sd=("Macro_Recall","std"),
        Accuracy_mean=("Accuracy","mean"),Accuracy_sd=("Accuracy","std"))
    for name, table in {
        "wine_metrics_raw":metrics,"wine_class_recall_raw":class_recall,
        "wine_fidelity_raw":fidelity,"wine_delta_component_raw":delta,
        "wine_validation_raw":validation,"wine_selected_lambda_raw":selected,
        "wine_validation_selected_test_raw":selected_test,
        "wine_validation_selected_test_summary":selected_summary
    }.items():
        table.to_csv(out/f"{name}.csv",index=False)
    print("PASSED: all expected row counts, selection joins and output saves")
    print("Results saved in:",out)
    print(selected_test[["RunID","Method","Classifier","Selected_Lambda","Macro_F1"]].to_string(index=False))

if __name__ == "__main__":
    main()


Mounted at /content/drive
Wine Quality (Red) (1599, 11) class counts {np.int64(0): np.int64(63), np.int64(1): np.int64(1319), np.int64(2): np.int64(217)}
Device: cuda Runs: 10
Run 1/10: CGAN


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Run 1/10: CWGAN-GP


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptro

Completed seed 44


...

Run 10/10: CWGAN-GP


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptro

Completed seed 53


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


PASSED: all expected row counts, selection joins and output saves
Results saved in: /content/drive/MyDrive/CGA Ex/revision_protocol
 RunID   Method Classifier  Selected_Lambda  Macro_F1
     0     CGAN        MLP             1.00  0.443353
     0     CGAN         RF             1.00  0.500191
     0 CWGAN-GP        MLP             0.25  0.446432
     0 CWGAN-GP         RF             0.25  0.489111
     0      ROS        MLP             1.00  0.481558
     0      ROS         RF             0.75  0.507119
     0    SMOTE        MLP             0.25  0.512131
     0    SMOTE         RF             0.50  0.570326
     1     CGAN        MLP             0.50  0.471583
     1     CGAN         RF             0.75  0.542989
     1 CWGAN-GP        MLP             0.25  0.425221
     1 CWGAN-GP         RF             0.50  0.502592
     1      ROS        MLP             1.00  0.491030
     1      ROS         RF             0.75  0.537710
     1    SMOTE        MLP             0.25  0.500362
    